# Arabizi Sentiment Analysis using LoRA Fine-Tuning

## Notebook Structure
1. Install Required Packages
2. Import Libraries
3. Configuration
4. Helper Functions
5. Load Dataset
6. Load Base Model & Tokenizer
7. Apply LoRA
8. Prepare Dataset
9. Baseline Evaluation
10. Fine-Tuning
11. Evaluation
12. Save Model
13. Inference
14. Results Export (writes evaluation_metrics.json, classification_report.txt,
    sample_predictions.csv, misclassified_examples.csv, training_history.csv,
    confusion_matrix.png, training_curves.png, dataset_statistics.json,
    training_config.json, and README.md to results/)


# LoRA Fine-Tuning: Arabizi Sentiment Classification
**Model:** MBZUAI-Paris/Nile-Chat-4B  
**Task:** 3-class sentiment (positive / negative / neutral)  
**Method:** LoRA (Low-Rank Adaptation) — trains < 1% of parameters, fits in 4-6 GB VRAM

**Notebook conventions:** run from the `notebooks/` folder; data/results/models are
referenced via relative paths to sibling project folders (`data/`, `results/`, `models/`).


## Running this on Kaggle

Before running, in the notebook editor sidebar:
1. **Settings > Accelerator**: choose a GPU (T4 x2 or P100).
2. **Settings > Internet**: turn **On** (needed to `pip install` and to download the base model from Hugging Face).
3. **Add Input**: upload `arabizi_data.csv` as a Kaggle Dataset (or attach an existing one), then click "Add Input" and attach it to this notebook. The code below auto-detects the file under `/kaggle/input/`, so the dataset folder name doesn't matter.
4. If `MBZUAI-Paris/Nile-Chat-4B` requires authentication, add a Kaggle **Secret** named `HF_TOKEN` with your Hugging Face access token (Add-ons > Secrets), and run the optional login cell below.

Outputs (adapter weights, plots) are written to `/kaggle/working/`, which you can download from the notebook's Output panel after the run finishes.


# 1. Install Packages

**Purpose:** 1. Install Packages section of the workflow.


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets scikit-learn matplotlib seaborn huggingface_hub


# 2. Import Packages

**Purpose:** 2. Import Packages section of the workflow.


In [ ]:
import csv
import json
import os
import random
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")


### (Optional) Hugging Face login
Only needed if `MBZUAI-Paris/Nile-Chat-4B` is gated. Skip if the model downloads without a token.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to Hugging Face using Kaggle secret HF_TOKEN.")
except Exception as e:
    print(f"Skipping HF login (no HF_TOKEN secret found or not needed): {e}")


# 3. Define Functions

All helper functions and dataset classes are defined here, in one place,
before they are used further down.

**Purpose:** 3. Define Functions

All helper functions and dataset classes are defined here, in one place,
before they are used further down. section of the workflow.


In [ ]:
def make_prompt(text, label_str=None, tokenizer=None):
    """Format as chat prompt. During training, append the label as the answer."""
    user_msg = (
        f'Classify the sentiment of this Egyptian Arabizi text as exactly one word: '
        f'positive, negative, or neutral.\n\nText: "{text}"\nAnswer:'
    )
    messages = [{"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    if label_str is not None:
        prompt += f" {label_str}"
    return prompt


In [ ]:
class ArabiziDataset(Dataset):
    """Tokenized prompt/label pairs for LoRA fine-tuning."""

    def __init__(self, texts, labels, tokenizer, id2label, max_length):
        self.samples = []
        for text, label in zip(texts, labels):
            label_str = id2label[label]
            prompt = make_prompt(text, label_str, tokenizer)
            enc = tokenizer(
                prompt,
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].squeeze()
            attention_mask = enc["attention_mask"].squeeze()

            # Labels: -100 everywhere except the answer token(s)
            label_ids = input_ids.clone()
            label_ids[attention_mask == 0] = -100

            # Compute prompt_len using the same tokenizer call style
            # (with special tokens, the default) so the mask offset matches.
            prompt_only = make_prompt(text, tokenizer=tokenizer)  # no label
            prompt_len = len(tokenizer(prompt_only, return_tensors="pt")["input_ids"][0])
            label_ids[:prompt_len] = -100

            self.samples.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": label_ids,
                "true_label": label,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


In [ ]:
def predict(model, tokenizer, texts_list, label2id):
    """Run greedy generation and map the decoded word back to a label id."""
    model.eval()
    preds = []
    # With device_map="auto" the model may span multiple devices.
    # Move inputs to the device of the embedding layer (always the first shard).
    input_device = next(model.parameters()).device
    with torch.no_grad():
        for text in texts_list:
            prompt = make_prompt(text, tokenizer=tokenizer)
            inputs = tokenizer(prompt, return_tensors="pt").to(input_device)
            out = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            new_tokens = out[0][inputs["input_ids"].shape[1]:]  # slice off the prompt
            if len(new_tokens) > 0:
                decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
                decoded = decoded.strip(".,!?\"'").split()[0] if decoded else ""
            else:
                decoded = ""
            pred_id = label2id.get(decoded, 1)  # default neutral if unrecognized
            preds.append(pred_id)
    return preds


# 4. Config

**Purpose:** 4. Config section of the workflow.


In [ ]:
# Project folders — auto-detects Kaggle vs. a local checkout.
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    RESULTS_DIR = "/kaggle/working/results"
    MODELS_DIR = "/kaggle/working/models"

    # Locate arabizi_data.csv wherever the attached dataset was mounted
    # (the exact /kaggle/input/<dataset-slug>/ folder name doesn't matter).
    DATA_PATH = None
    for root, _dirs, files in os.walk("/kaggle/input"):
        if "arabizi_data.csv" in files:
            DATA_PATH = os.path.join(root, "arabizi_data.csv")
            break
    if DATA_PATH is None:
        raise FileNotFoundError(
            "Could not find arabizi_data.csv under /kaggle/input. "
            "Attach it via 'Add Input' in the Kaggle notebook editor."
        )
    DATA_DIR = os.path.dirname(DATA_PATH)
else:
    # Local: this notebook runs from notebooks/, so paths are relative to it
    DATA_DIR = "../data"
    RESULTS_DIR = "../results"
    MODELS_DIR = "../models"
    DATA_PATH = os.path.join(DATA_DIR, "arabizi_data.csv")  # columns: text, label

OUTPUT_DIR = os.path.join(MODELS_DIR, "lora_adapter")
MODEL_ID = "MBZUAI-Paris/Nile-Chat-4B"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Training hyperparameters
MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 5
LR = 2e-4
TEST_SIZE = 0.2  # 80% train, 20% test
SEED = 42

# LoRA hyperparameters
LORA_R = 8  # rank - higher = more capacity, more VRAM
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
training_config = {
    "model_id": MODEL_ID,
    "task": "3-class sentiment (positive / negative / neutral)",
    "method": "LoRA (Low-Rank Adaptation)",
    "quantization": "4-bit NF4 (bitsandbytes)",
    "hyperparameters": {
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "test_size": TEST_SIZE,
        "seed": SEED,
    },
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": ["q_proj", "v_proj"],
    },
    "label_map": LABEL2ID,
    "environment": "kaggle" if IS_KAGGLE else "local",
    "data_path": DATA_PATH,
    "output_dir": OUTPUT_DIR,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

training_config_path = os.path.join(RESULTS_DIR, "training_config.json")
with open(training_config_path, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)
print(f"Saved -> {training_config_path}")


# 5. Load & Split Data

**Purpose:** 5. Load & Split Data section of the workflow.


In [ ]:
print(f"Files in {DATA_DIR}:")
for fname in os.listdir(DATA_DIR):
    print(f"  {fname}")


In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    first_lines = [next(f) for _ in range(3)]
for line in first_lines:
    print(repr(line))


In [ ]:
texts, labels = [], []
with open(DATA_PATH, encoding="utf-8-sig") as f:  # utf-8-sig strips the BOM
    reader = csv.DictReader(f)  # regular comma separator
    for row in reader:
        label = row["label"].strip().lower()
        if label in LABEL2ID:
            texts.append(row["text"].strip())
            labels.append(LABEL2ID[label])

print(f"Total examples: {len(texts)}")
for name, idx in LABEL2ID.items():
    print(f"  {name}: {labels.count(idx)}")

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)
print(f"\nTrain: {len(train_texts)} | Test: {len(test_texts)}")


In [ ]:
def _length_stats(text_list):
    lengths = [len(t.split()) for t in text_list]
    return {
        "min_words": int(np.min(lengths)) if lengths else 0,
        "max_words": int(np.max(lengths)) if lengths else 0,
        "mean_words": float(np.mean(lengths)) if lengths else 0.0,
        "median_words": float(np.median(lengths)) if lengths else 0.0,
    }

dataset_statistics = {
    "data_path": DATA_PATH,
    "total_examples": len(texts),
    "class_distribution": {
        name: labels.count(idx_) for name, idx_ in LABEL2ID.items()
    },
    "class_distribution_pct": {
        name: round(100 * labels.count(idx_) / len(labels), 2) if labels else 0.0
        for name, idx_ in LABEL2ID.items()
    },
    "train_size": len(train_texts),
    "test_size": len(test_texts),
    "test_split_fraction": TEST_SIZE,
    "train_class_distribution": {
        name: train_labels.count(idx_) for name, idx_ in LABEL2ID.items()
    },
    "test_class_distribution": {
        name: test_labels.count(idx_) for name, idx_ in LABEL2ID.items()
    },
    "text_length_words": {
        "overall": _length_stats(texts),
        "train": _length_stats(train_texts),
        "test": _length_stats(test_texts),
    },
}

dataset_statistics_path = os.path.join(RESULTS_DIR, "dataset_statistics.json")
with open(dataset_statistics_path, "w", encoding="utf-8") as f:
    json.dump(dataset_statistics, f, indent=2, ensure_ascii=False)
print(f"Saved -> {dataset_statistics_path}")


# 6. Load Tokenizer & Model (4-bit Quantized)

**Purpose:** 6. Load Tokenizer & Model (4-bit Quantized) section of the workflow.


In [ ]:
# 4-bit quantization - loads the 4B model in ~3 GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.pad_token_id = tok.pad_token_id

# Required before applying LoRA to a quantized model
base_model = prepare_model_for_kbit_training(base_model)
print("Model loaded and prepared for k-bit training.")


# 7. Attach LoRA Adapter

**Purpose:** 7. Attach LoRA Adapter section of the workflow.


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],  # attention layers only
    bias="none",
)

model = get_peft_model(base_model, lora_config)

# Allow non-LoRA frozen layers to pass gradients through
model.enable_input_require_grads()

model.print_trainable_parameters()  # should be ~0.1-1% of total


# 8. Build Datasets & Loaders

**Purpose:** 8. Build Datasets & Loaders section of the workflow.


In [ ]:
train_dataset = ArabiziDataset(train_texts, train_labels, tok, ID2LABEL, MAX_LENGTH)
test_dataset = ArabiziDataset(test_texts, test_labels, tok, ID2LABEL, MAX_LENGTH)

# num_workers=0 avoids CUDA multiprocessing errors with quantized models
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=0)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


# 9. Baseline F1 (Before Training)

**Purpose:** 9. Baseline F1 (Before Training) section of the workflow.


In [ ]:
print("Running baseline inference on test set (this takes a few minutes)...")
baseline_preds = predict(model, tok, test_texts, LABEL2ID)
baseline_f1 = f1_score(test_labels, baseline_preds, average="macro")
print(f"\nBaseline Macro F1 (before LoRA training): {baseline_f1:.4f}")
print(classification_report(test_labels, baseline_preds, target_names=list(LABEL2ID.keys())))


# 10. Training Loop

**Purpose:** 10. Training Loop section of the workflow.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

train_losses = []
epoch_f1s = []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        model_device = next(model.parameters()).device
        input_ids = batch["input_ids"].to(model_device)
        attention_mask = batch["attention_mask"].to(model_device)
        labels_b = batch["labels"].to(model_device)

        # zero_grad BEFORE the forward pass, not after backward
        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_b)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Evaluate
    preds = predict(model, tok, test_texts, LABEL2ID)
    f1 = f1_score(test_labels, preds, average="macro")
    epoch_f1s.append(f1)

    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Macro F1: {f1:.4f}")

print("\nTraining complete.")


In [ ]:
training_history_path = os.path.join(RESULTS_DIR, "training_history.csv")
with open(training_history_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "macro_f1", "baseline_macro_f1"])
    writer.writerow([0, "", round(baseline_f1, 6), round(baseline_f1, 6)])
    for epoch, (loss, f1) in enumerate(zip(train_losses, epoch_f1s), start=1):
        writer.writerow([epoch, round(loss, 6), round(f1, 6), round(baseline_f1, 6)])
print(f"Saved -> {training_history_path}")


# 11. Results & Plots

**Purpose:** 11. Results & Plots section of the workflow.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(range(1, EPOCHS + 1), train_losses, marker="o", color="steelblue")
axes[0].set_title("Training Loss", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# F1 curve
axes[1].axhline(baseline_f1, color="gray", linestyle="--", label=f"Baseline: {baseline_f1:.3f}")
axes[1].plot(range(1, EPOCHS + 1), epoch_f1s, marker="o", color="darkorange", label="After LoRA")
axes[1].set_title("Macro F1 per Epoch", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
training_curves_path = os.path.join(RESULTS_DIR, "training_curves.png")
plt.savefig(training_curves_path, dpi=150)
plt.show()
print(f"Saved -> {training_curves_path}")


In [ ]:
# Final classification report
final_preds = predict(model, tok, test_texts, LABEL2ID)
final_f1 = f1_score(test_labels, final_preds, average="macro")

print("=" * 50)
print(f"  Baseline Macro F1 : {baseline_f1:.4f}")
print(f"  Final    Macro F1 : {final_f1:.4f}")
print(f"  Improvement       : +{final_f1 - baseline_f1:.4f}")
print("=" * 50)
print(classification_report(test_labels, final_preds, target_names=list(LABEL2ID.keys())))


In [ ]:
# --- classification_report.txt (human-readable) ---
report_text = classification_report(
    test_labels, final_preds, target_names=list(LABEL2ID.keys())
)
classification_report_path = os.path.join(RESULTS_DIR, "classification_report.txt")
with open(classification_report_path, "w", encoding="utf-8") as f:
    f.write("Arabizi Sentiment Classification — Final Evaluation\n")
    f.write("=" * 55 + "\n")
    f.write(f"Model: {MODEL_ID}\n")
    f.write(f"Baseline Macro F1: {baseline_f1:.4f}\n")
    f.write(f"Final Macro F1:    {final_f1:.4f}\n")
    f.write(f"Improvement:       +{final_f1 - baseline_f1:.4f}\n")
    f.write("=" * 55 + "\n\n")
    f.write(report_text)
print(f"Saved -> {classification_report_path}")

# --- evaluation_metrics.json (machine-readable) ---
report_dict = classification_report(
    test_labels, final_preds, target_names=list(LABEL2ID.keys()), output_dict=True
)
evaluation_metrics = {
    "baseline_macro_f1": baseline_f1,
    "final_macro_f1": final_f1,
    "improvement": final_f1 - baseline_f1,
    "per_class": {
        name: report_dict[name] for name in LABEL2ID.keys()
    },
    "accuracy": report_dict.get("accuracy"),
    "macro_avg": report_dict.get("macro avg"),
    "weighted_avg": report_dict.get("weighted avg"),
    "test_size": len(test_labels),
}
evaluation_metrics_path = os.path.join(RESULTS_DIR, "evaluation_metrics.json")
with open(evaluation_metrics_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_metrics, f, indent=2, ensure_ascii=False)
print(f"Saved -> {evaluation_metrics_path}")


In [ ]:
pred_label_names = [ID2LABEL[p] for p in final_preds]
true_label_names = [ID2LABEL[t] for t in test_labels]
correct_flags = [p == t for p, t in zip(final_preds, test_labels)]

# --- sample_predictions.csv: a random sample of test-set predictions ---
sample_size = min(50, len(test_texts))
rng = random.Random(SEED)
sample_idx = sorted(rng.sample(range(len(test_texts)), sample_size))

sample_predictions_path = os.path.join(RESULTS_DIR, "sample_predictions.csv")
with open(sample_predictions_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "true_label", "predicted_label", "correct"])
    for i in sample_idx:
        writer.writerow(
            [test_texts[i], true_label_names[i], pred_label_names[i], correct_flags[i]]
        )
print(f"Saved -> {sample_predictions_path} ({sample_size} rows)")

# --- misclassified_examples.csv: every wrong prediction on the test set ---
misclassified_examples_path = os.path.join(RESULTS_DIR, "misclassified_examples.csv")
with open(misclassified_examples_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "true_label", "predicted_label"])
    n_wrong = 0
    zipped = zip(test_texts, true_label_names, pred_label_names, correct_flags)
    for text, true_name, pred_name, correct in zipped:
        if not correct:
            writer.writerow([text, true_name, pred_name])
            n_wrong += 1
print(
    f"Saved -> {misclassified_examples_path} "
    f"({n_wrong} misclassified of {len(test_texts)} test examples)"
)


In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, final_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=list(LABEL2ID.keys()),
    yticklabels=list(LABEL2ID.keys()),
)
plt.title(f"Confusion Matrix (Macro F1={final_f1:.3f})", fontsize=13)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
confusion_matrix_path = os.path.join(RESULTS_DIR, "confusion_matrix.png")
plt.savefig(confusion_matrix_path, dpi=150)
plt.show()
print(f"Saved -> {confusion_matrix_path}")


# 12. Save the LoRA Adapter

**Purpose:** 12. Save the LoRA Adapter section of the workflow.


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved -> {OUTPUT_DIR}")


# 13. Inference with Trained Adapter

**Purpose:** 13. Inference with Trained Adapter section of the workflow.


In [ ]:
test_sentences = [
    "el film da kwayes awi, mestanyaha tani",
    "el akl kan wehesh awi, mesh hanraga3 tani",
    "msh 3aref a2ol 7aga, el makan 3ady",
    # add your own here
]

print("Predictions with trained LoRA adapter:\n")
for sent in test_sentences:
    pred_id = predict(model, tok, [sent], LABEL2ID)[0]
    print(f"  '{sent}'")
    print(f"  -> {ID2LABEL[pred_id]}\n")


# Summary

- Notebook organized into logical sections.
- Relative paths used for data, models, and results.
- Fine-tuned LoRA adapter saved for inference.
- Notebook follows a clearer academic structure for submission.


## Results Export

Write a `README.md` summarizing the run and confirm every results file listed above has been created.


In [ ]:
readme_content = f"""# Results — Arabizi Sentiment LoRA Fine-Tuning

**Model:** {MODEL_ID}
**Method:** LoRA (r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}) on a 4-bit quantized base model
**Generated:** {datetime.now(timezone.utc).isoformat()}
**Environment:** {"Kaggle" if IS_KAGGLE else "Local"}

## Headline results

| Metric | Value |
|---|---|
| Baseline Macro F1 (before LoRA) | {baseline_f1:.4f} |
| Final Macro F1 (after LoRA) | {final_f1:.4f} |
| Improvement | +{final_f1 - baseline_f1:.4f} |
| Train / Test examples | {len(train_texts)} / {len(test_texts)} |
| Epochs / Batch size / LR | {EPOCHS} / {BATCH_SIZE} / {LR} |

## Files in this folder

| File | Description |
|---|---|
| `evaluation_metrics.json` | Final precision/recall/F1 per class, macro & weighted averages, baseline vs. final comparison |
| `classification_report.txt` | Human-readable sklearn classification report with summary header |
| `sample_predictions.csv` | Random sample of {min(50, len(test_texts))} test-set predictions (text, true label, predicted label, correct) |
| `misclassified_examples.csv` | Every test example the fine-tuned model got wrong |
| `training_history.csv` | Per-epoch training loss and macro F1, plus the pre-training baseline |
| `confusion_matrix.png` | Confusion matrix heatmap for the final model on the test set |
| `training_curves.png` | Training loss curve and macro F1 curve across epochs |
| `dataset_statistics.json` | Class distribution, train/test split sizes, and text length stats |
| `training_config.json` | Full hyperparameter and run configuration used for this training run |
| `README.md` | This file |

## Reproducing this run

Data source: `{DATA_PATH}`
Random seed: {SEED}
LoRA adapter saved to: `{OUTPUT_DIR}`
"""

readme_path = os.path.join(RESULTS_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)
print(f"Saved -> {readme_path}")

print("\nFinal results/ contents:")
for fname in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {fname}")
